# Bring-Your-Own-Container (BYOC)

The most control, the most responsibility: SageMaker no longer manages any part of the
container. You own the runtime, dependencies, training script, and serving stack — SageMaker
just orchestrates compute, moves data in/out of S3, and calls `docker run <image> train` /
`docker run <image> serve` at the right times.

The BYOC training contract SageMaker enforces:
```
/opt/ml/input/data/<channel>/              input channels (train, validation, ...)
/opt/ml/input/config/hyperparameters.json  hyperparameters (all values arrive as strings)
/opt/ml/model/                             write the trained model artifact here
```

## 1. Build and push the image to ECR

Run `setup.sh`, which: creates the ECR repo if it doesn't exist, `docker build`s the image
from the `Dockerfile` here (`train.py` copied in as the executable `/usr/local/bin/train`,
matching the contract above), then tags and pushes it. On Apple Silicon, note the explicit
`--platform linux/amd64` in the script — SageMaker's training instances are x86, so an
unspecified build would produce an incompatible arm64 image.

## 2. Train against the custom image

In [ ]:
import boto3
import sagemaker
from sagemaker.estimator import Estimator

#TODO: Update the following variables with your own values
REGION = ""
BUCKET_NAME = ""
EXECUTION_ROLE_ARN = ""
IMAGE_URI = ""

# ----------------------

boto_session = boto3.Session(region_name=REGION)
sm_session = sagemaker.Session(boto_session=boto_session)

# Reusing the train/validation data already uploaded to S3
train_uri = f"s3://{BUCKET_NAME}/processed/script_mode/train/train.csv"
val_uri = f"s3://{BUCKET_NAME}/processed/script_mode/val/validation.csv"

estimator = Estimator(
    image_uri=IMAGE_URI,
    role=EXECUTION_ROLE_ARN,
    instance_count=1,
    instance_type="ml.m5.large",
    output_path=f"s3://{BUCKET_NAME}/models/byoc/",
    sagemaker_session=sm_session,
)

# Note: with BYOC there's no SDK hyperparameter validation -- these land in
# /opt/ml/input/config/hyperparameters.json exactly as train.py expects.

estimator.set_hyperparameters(max_depth=3, eta=0.2, num_round=75, scale_pos_weight=770)

# When estimator.fit is called, multiple things get triggered - 
# 1. AWS provisions the instance which we've mentioned above
# 2. TrainingJob gets created
# 3. Docker image, training/validation data from S3 gets downloaded
# 4. hyperparameter, config files etc get created
# 5. docker image start and relevant config files are mounted....
estimator.fit({"train": train_uri, "validation": val_uri})